<a href="https://colab.research.google.com/github/f171p/323project/blob/main/HighScoreList.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import List, Tuple, Optional
import csv
import os
from functools import reduce

#High scores csv file
FILE_PATH = "/content/sudoku_highscores.csv" # @param {type:"string"}

# HighScore-Datentyp als immutable Tuple
HighScore = Tuple[str, str, str, int]  # (Name, Datum, Level, Zeit)

In [ ]:
class HighScoreAdmin:
  def __init__(self, file_path: str):
        self.file_path = file_path
        self.highscores = self.load_scores()

  def load_scores(self) -> dict:
        if not os.path.exists(self.file_path):
            return {"einfach": [], "mittel": [], "schwer": [], "genie": []}

        with open(self.file_path, newline='', encoding='ISO-8859-1') as csvfile:
            reader = csv.DictReader(csvfile, delimiter=',')  # Tab-separierte Datei
            data = [(row["Spielername"], row["Datum"], row["Level"].lower(), int(row["Benötigte Zeit (Sekunden)"])) for row in reader]
            return reduce(lambda acc, row: self.insert_score(acc, row), data, {"einfach": [], "mittel": [], "schwer": [], "genie": []})

  def save_scores(self):
        with open(self.file_path, mode='w', newline='', encoding='ISO-8859-1') as csvfile:
            fieldnames = ["Spielername", "Datum", "Level", "Benötigte Zeit (Sekunden)"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames, delimiter=',')
            writer.writeheader()
            for level, scores in self.highscores.items():
                for score in scores:
                    writer.writerow({"Spielername": score[0], "Datum": score[1], "Level": score[2], "Benötigte Zeit (Sekunden)": score[3]})

  def insert_score(self, scores: dict, score: HighScore) -> dict:
        level = score[2]
        scores[level] = sorted(scores.get(level, []) + [score], key=lambda x: x[3])[:10]
        return scores

  def get_rank(self, level: str, time: int) -> Optional[int]:
        level_scores = self.highscores.get(level, [])
        if len(level_scores) < 10 or time < level_scores[-1][3]:
            return next((i + 1 for i, (_, _, _, t) in enumerate(level_scores) if time < t), len(level_scores) + 1)
        return None

  def display_highscores(self, level: str):
        return "\n".join(f"{i + 1}. {name} - {date} - {time}s" for i, (name, date, _, time) in enumerate(self.highscores.get(level, [])))

  def add_score(self, name: str, date: str, level: str, time: int):
        rank = self.get_rank(level, time)
        if rank:
            self.highscores = self.insert_score(self.highscores, (name, date, level, time))
            return f"Your rank: {rank}!"
        return f"HighScore entries only better than {self.highscores[level][-1][3]} seconds"

def menu(actions: list):
  admin = HighScoreAdmin(FILE_PATH)
  for action in actions:
      if action[0] == "load":
          print("Loading highscores…")
          admin.load_scores()
          print("Loaded entries:", len(admin.highscores))
      elif action[0] == "add":
          print(admin.add_score(action[1], action[2], action[3], action[4]))
      elif action[0] == "display":
            print(admin.display_highscores(action[1]))
      elif action[0] == "save":
            admin.save_scores()
            print("Highscores gespeichert.")

In [ ]:
test_actions = [
    ("load",),
    ("display", "genie")
]
menu(test_actions)

Loading highscores…
Loaded entries: 4

